# Fantasy Football Player Breakout Prediction - Procedural Implementation

This notebook provides a fully procedural implementation of the fantasy football player breakout prediction pipeline. All functionality is defined within the notebook cells themselves, without importing functions from external modules.

The pipeline predicts which fantasy football players are likely to have breakout seasons (30%+ fantasy point increases by default) using historical NFL data from nflverse. Each step is transparent and customizable, making the process more accessible for analysis and modification.

## Setup and Imports

Import all necessary libraries and helper functions:

In [ ]:
# Core data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# Additional imports needed for data loading
import ssl
from urllib.request import urlopen
from typing import Dict, List, Tuple, Optional

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ All imports successful!")

## Configuration

Set up pipeline parameters:

In [ ]:
# Pipeline configuration
seasons = list(range(2018, 2025))  # Historical seasons to analyze
breakout_threshold = 0.3  # 30% fantasy point increase threshold
predict_season = 2025     # Season to predict breakouts for
test_size = 0.2          # Train/test split ratio
random_state = 42        # For reproducibility

print(f"📊 Configuration:")
print(f"   • Analyzing seasons: {min(seasons)}-{max(seasons)}")
print(f"   • Breakout threshold: {breakout_threshold*100}% fantasy point increase")
print(f"   • Predicting for season: {predict_season}")
print(f"   • Test size: {test_size*100}%")

## Helper Functions Definition

Define the core functions needed for the breakout prediction pipeline procedurally within this notebook:

In [ ]:
# URLs for nflverse data
PLAYER_STATS_URL = "https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats.csv"
ROSTER_URL = "https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_{season}.csv"
INJURY_URL = "https://github.com/nflverse/nflverse-data/releases/download/injuries/injuries_{season}.csv"

print("📊 Data source URLs configured")

In [ ]:
def load_player_data(seasons: List[int]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load player stats and roster data for multiple seasons."""
    ctx = ssl._create_unverified_context()
    
    # Load player stats
    player_stats = pd.read_csv(urlopen(PLAYER_STATS_URL, context=ctx))
    
    # Filter to regular season only and relevant seasons
    player_stats = player_stats[
        (player_stats['season_type'] == 'REG') & 
        (player_stats['season'].isin(seasons))
    ].copy()
    
    # Load roster data for team changes and experience
    rosters = []
    for season in seasons:
        try:
            roster_url = ROSTER_URL.format(season=season)
            roster = pd.read_csv(urlopen(roster_url, context=ctx))
            roster['season'] = season
            rosters.append(roster)
        except Exception:
            continue
    
    roster_data = pd.concat(rosters, ignore_index=True) if rosters else pd.DataFrame()
    
    return player_stats, roster_data

print("✅ load_player_data function defined")

In [ ]:
def aggregate_season_stats(player_stats: pd.DataFrame) -> pd.DataFrame:
    """Aggregate weekly stats to season-level for each player."""
    # Filter out rows with null player names
    player_stats = player_stats[player_stats['player_name'].notna()].copy()
    
    # Group by player and season, sum relevant fantasy stats
    agg_dict = {
        'fantasy_points': 'sum',
        'fantasy_points_ppr': 'sum',
        'carries': 'sum',
        'rushing_yards': 'sum',
        'rushing_tds': 'sum',
        'targets': 'sum',
        'receptions': 'sum',
        'receiving_yards': 'sum',
        'receiving_tds': 'sum',
        'passing_yards': 'sum',
        'passing_tds': 'sum',
        'interceptions': 'sum',
        'week': 'count',  # Count of weeks = games played
        'position': 'first',
        'recent_team': 'first'
    }
    
    season_stats = player_stats.groupby(['player_name', 'season']).agg(agg_dict).reset_index()
    
    # Rename week count to games_played
    season_stats.rename(columns={'week': 'games_played'}, inplace=True)
    
    # Filter to players with meaningful playing time (lowered threshold to include more data)
    season_stats = season_stats[season_stats['games_played'] >= 2].copy()
    
    return season_stats

print("✅ aggregate_season_stats function defined")

In [ ]:
def calculate_breakouts(season_stats: pd.DataFrame, breakout_threshold: float = 0.3) -> pd.DataFrame:
    """Calculate year-over-year breakouts in fantasy performance."""
    # Sort by player and season
    season_stats = season_stats.sort_values(['player_name', 'season'])
    
    # Calculate previous season stats
    season_stats['prev_fantasy_points'] = season_stats.groupby('player_name')['fantasy_points'].shift(1)
    season_stats['prev_fantasy_points_ppr'] = season_stats.groupby('player_name')['fantasy_points_ppr'].shift(1)
    season_stats['prev_team'] = season_stats.groupby('player_name')['recent_team'].shift(1)
    season_stats['prev_games'] = season_stats.groupby('player_name')['games_played'].shift(1)
    
    # Include players with lower previous production for breakout potential (was 20, now 10)
    # Breakouts often come from previously lower-scoring players
    meaningful_players = season_stats['prev_fantasy_points'] >= 10
    
    # Calculate percentage change in fantasy points
    season_stats['fantasy_change'] = (
        (season_stats['fantasy_points'] - season_stats['prev_fantasy_points']) / 
        season_stats['prev_fantasy_points']
    )
    season_stats['fantasy_ppr_change'] = (
        (season_stats['fantasy_points_ppr'] - season_stats['prev_fantasy_points_ppr']) / 
        season_stats['prev_fantasy_points_ppr']
    )
    
    # Define breakout (binary target) - for meaningful players
    season_stats['breakout'] = 0
    season_stats.loc[meaningful_players, 'breakout'] = (
        season_stats.loc[meaningful_players, 'fantasy_change'] > breakout_threshold
    ).astype(int)
    
    # Team change indicator (can indicate new opportunity)
    season_stats['team_change'] = (season_stats['recent_team'] != season_stats['prev_team']).astype(int)
    
    # Remove first season for each player and players without meaningful previous production
    season_stats = season_stats.dropna(subset=['prev_fantasy_points']).copy()
    season_stats = season_stats[meaningful_players].copy()
    
    return season_stats

print("✅ calculate_breakouts function defined")

In [ ]:
def engineer_features(season_stats: pd.DataFrame, roster_data: pd.DataFrame) -> pd.DataFrame:
    """Engineer features for breakout prediction."""
    # Merge with roster data to get age/experience info
    roster_subset = roster_data[['full_name', 'season', 'years_exp', 'birth_date']].copy()
    roster_subset.rename(columns={'full_name': 'player_name'}, inplace=True)
    
    merged = season_stats.merge(roster_subset, on=['player_name', 'season'], how='left')
    
    # Calculate age if birth_date is available
    merged['birth_date'] = pd.to_datetime(merged['birth_date'], errors='coerce')
    merged['age'] = merged.apply(
        lambda row: (pd.Timestamp(f'{row["season"]}-09-01') - row['birth_date']).days / 365.25
        if pd.notna(row['birth_date']) else np.nan, axis=1
    )
    
    # Previous season workload features (lower workload might indicate opportunity)
    merged['prev_workload'] = merged['prev_fantasy_points'] / merged.groupby('player_name')['prev_fantasy_points'].transform('max')
    
    # Breakout-specific features
    # Low previous production relative to position average (indicates upside potential)
    position_avg = merged.groupby(['position', 'season'])['prev_fantasy_points'].transform('mean')
    merged['prev_below_position_avg'] = (merged['prev_fantasy_points'] < position_avg).astype(int)
    
    # Young player indicator (more breakout potential)
    merged['is_young'] = (merged['years_exp'] <= 3).astype(int)
    
    # Position-specific features
    position_dummies = pd.get_dummies(merged['position'], prefix='pos')
    merged = pd.concat([merged, position_dummies], axis=1)
    
    # Fill missing values
    merged['years_exp'] = merged['years_exp'].fillna(merged['years_exp'].median())
    merged['age'] = merged['age'].fillna(merged['age'].median())
    
    return merged

print("✅ engineer_features function defined")

In [ ]:
def select_features(df: pd.DataFrame) -> List[str]:
    """Select features for the model."""
    feature_columns = [
        'prev_fantasy_points', 'prev_fantasy_points_ppr', 'years_exp', 'age',
        'team_change', 'prev_workload', 'games_played', 'prev_below_position_avg', 'is_young'
    ]
    
    # Add position dummies
    pos_columns = [col for col in df.columns if col.startswith('pos_')]
    feature_columns.extend(pos_columns)
    
    # Only return features that exist in the dataframe
    return [col for col in feature_columns if col in df.columns]

print("✅ select_features function defined")
print("\n🎯 All helper functions are now defined procedurally within the notebook!")

## Step 1: Data Loading

Load raw player statistics and roster data from nflverse:

In [ ]:
print("📥 Loading player data from nflverse...")
player_stats, roster_data = load_player_data(seasons)

print(f"✅ Data loaded successfully:")
print(f"   • Player stats: {len(player_stats):,} records")
print(f"   • Roster data: {len(roster_data):,} records")
print(f"   • Seasons covered: {sorted(player_stats['season'].unique())}")
print(f"   • Positions: {sorted(player_stats['position'].unique())}")

# Show sample of raw data
print("\n📋 Sample player stats:")
print(player_stats.head())

## Step 2: Season Stats Aggregation

Aggregate weekly player performance to season-level statistics:

In [ ]:
print("🔄 Aggregating weekly stats to season level...")
season_stats = aggregate_season_stats(player_stats)

print(f"✅ Season aggregation complete:")
print(f"   • Player-seasons: {len(season_stats):,}")
print(f"   • Unique players: {season_stats['player_name'].nunique():,}")
print(f"   • Columns: {len(season_stats.columns)}")

# Show sample aggregated data
print("\n📋 Sample aggregated season stats:")
print(season_stats[['player_name', 'season', 'position', 'fantasy_points', 'games']].head())

# Show fantasy points distribution
print("\n📈 Fantasy points distribution:")
print(season_stats['fantasy_points'].describe())

## Step 3: Breakout Calculation

Identify historical breakout seasons based on year-over-year fantasy point increases:

In [ ]:
print(f"🚀 Calculating breakouts (>{breakout_threshold*100}% increase)...")
breakout_data = calculate_breakouts(season_stats, breakout_threshold)

print(f"✅ Breakout calculation complete:")
print(f"   • Total player-seasons: {len(breakout_data):,}")
print(f"   • Breakout seasons: {breakout_data['breakout'].sum():,}")
print(f"   • Breakout rate: {breakout_data['breakout'].mean():.1%}")

# Show breakout distribution by position
print("\n📊 Breakouts by position:")
breakout_by_pos = breakout_data.groupby('position')['breakout'].agg(['count', 'sum', 'mean'])
breakout_by_pos.columns = ['Total', 'Breakouts', 'Rate']
breakout_by_pos['Rate'] = breakout_by_pos['Rate'].map('{:.1%}'.format)
print(breakout_by_pos)

# Show sample breakout data
print("\n📋 Sample players with breakouts:")
breakout_examples = breakout_data[breakout_data['breakout'] == 1].head()
print(breakout_examples[['player_name', 'season', 'position', 'fantasy_points', 'prev_fantasy_points', 'fantasy_change']].round(1))

## Step 4: Feature Engineering

Create predictive features from player stats and roster data:

In [ ]:
print("🔧 Engineering features for machine learning...")
feature_data = engineer_features(breakout_data, roster_data)

print(f"✅ Feature engineering complete:")
print(f"   • Total features: {len(feature_data.columns)}")
print(f"   • Training samples: {len(feature_data):,}")
print(f"   • Missing values: {feature_data.isnull().sum().sum():,}")

# Show available features
print("\n🎯 Available features:")
feature_columns = [col for col in feature_data.columns if col not in ['player_name', 'season', 'breakout']]
for i, col in enumerate(feature_columns, 1):
    print(f"{i:2d}. {col}")
    if i >= 15:  # Show first 15 features
        print(f"    ... and {len(feature_columns)-15} more")
        break

# Show sample feature data
print("\n📋 Sample feature data:")
sample_cols = ['player_name', 'season', 'position', 'age', 'experience', 'fantasy_points', 'breakout']
print(feature_data[sample_cols].head())

## Step 5: Feature Selection

Select the most relevant features for modeling:

In [ ]:
print("🎯 Selecting relevant features...")
selected_features = select_features(feature_data)

print(f"✅ Feature selection complete:")
print(f"   • Selected features: {len(selected_features)}")
print(f"   • Excluded: {len([col for col in feature_data.columns if col not in selected_features + ['player_name', 'season', 'breakout']])} features")

print("\n🎯 Selected features for modeling:")
for i, feature in enumerate(selected_features, 1):
    print(f"{i:2d}. {feature}")

## Step 6: Data Preparation for Modeling

Prepare features and target variables for machine learning:

In [ ]:
print("🔄 Preparing data for modeling...")

# Extract features and target
X = feature_data[selected_features].fillna(0)
y = feature_data['breakout']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

# Initialize and fit the scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data preparation complete:")
print(f"   • Training samples: {len(X_train):,} ({y_train.sum():,} breakouts, {y_train.mean():.1%} rate)")
print(f"   • Test samples: {len(X_test):,} ({y_test.sum():,} breakouts, {y_test.mean():.1%} rate)")
print(f"   • Features: {X_train.shape[1]}")
print(f"   • Feature scaling: Applied StandardScaler")

## Step 7: Model Training

Train a Random Forest classifier to predict breakouts:

In [ ]:
print("🤖 Training Random Forest model...")

# Initialize model with the same parameters as the original pipeline
model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=random_state
)

# Train the model
model.fit(X_train_scaled, y_train)

print("✅ Model training complete!")
print(f"   • Algorithm: Random Forest")
print(f"   • Trees: {model.n_estimators}")
print(f"   • Max depth: {model.max_depth}")
print(f"   • Training samples: {len(X_train_scaled):,}")

## Step 8: Model Evaluation

Evaluate model performance on training and test sets:

In [ ]:
print("📊 Evaluating model performance...")

# Make predictions
train_pred = model.predict(X_train_scaled)
test_pred = model.predict(X_test_scaled)
train_pred_proba = model.predict_proba(X_train_scaled)[:, 1]
test_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
train_auc = roc_auc_score(y_train, train_pred_proba)
test_auc = roc_auc_score(y_test, test_pred_proba)
train_ap = average_precision_score(y_train, train_pred_proba)
test_ap = average_precision_score(y_test, test_pred_proba)

print(f"✅ Model Performance:")
print(f"   • Training AUC: {train_auc:.3f}")
print(f"   • Test AUC: {test_auc:.3f}")
print(f"   • Training AP: {train_ap:.3f}")
print(f"   • Test AP: {test_ap:.3f}")

print("\n📋 Test Set Classification Report:")
print(classification_report(y_test, test_pred))

# Store results for later use
model_results = {
    'train_auc': train_auc,
    'test_auc': test_auc,
    'train_ap': train_ap,
    'test_ap': test_ap,
    'train_report': classification_report(y_train, train_pred),
    'test_report': classification_report(y_test, test_pred),
    'feature_importance': dict(zip(selected_features, model.feature_importances_))
}

## Step 9: Feature Importance Analysis

Understand which features are most predictive of breakouts:

In [ ]:
print("🎯 Analyzing feature importance...")

# Get feature importances
feature_importance = model_results['feature_importance']
features = list(feature_importance.keys())
importances = list(feature_importance.values())

# Sort by importance
sorted_idx = sorted(range(len(importances)), key=lambda i: importances[i], reverse=True)
sorted_features = [features[i] for i in sorted_idx]
sorted_importances = [importances[i] for i in sorted_idx]

# Plot top features
plt.figure(figsize=(12, 8))
plt.barh(sorted_features[:15], sorted_importances[:15])
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features for Predicting Fantasy Breakouts')
plt.tight_layout()
plt.show()

# Print top features
print("\n🏆 Top 10 Feature Importances:")
for i, (feature, importance) in enumerate(zip(sorted_features[:10], sorted_importances[:10]), 1):
    print(f"{i:2d}. {feature}: {importance:.3f}")

## Step 10: Generate Predictions

Make breakout predictions for the target season:

In [ ]:
print(f"🔮 Generating predictions for {predict_season}...")

# Get most recent season data for each player
current_data = feature_data[feature_data['season'] == predict_season - 1].copy()

if current_data.empty:
    print(f"❌ No data available for season {predict_season - 1}")
else:
    # Prepare features for prediction
    X_current = current_data[selected_features].fillna(0)
    X_current_scaled = scaler.transform(X_current)
    
    # Make predictions
    breakout_proba = model.predict_proba(X_current_scaled)[:, 1]
    breakout_pred = model.predict(X_current_scaled)
    
    # Create predictions dataframe
    predictions = current_data[['player_name', 'position', 'recent_team', 'fantasy_points']].copy()
    predictions['breakout_probability'] = breakout_proba
    predictions['predicted_breakout'] = breakout_pred
    
    # Add potential tiers
    predictions['potential_tier'] = pd.cut(
        breakout_proba, 
        bins=[0, 0.3, 0.6, 1.0], 
        labels=['Low Potential', 'Medium Potential', 'High Potential']
    )
    
    # Sort by probability
    predictions = predictions.sort_values('breakout_probability', ascending=False)
    
    print(f"✅ Predictions generated for {len(predictions):,} players")
    print(f"   • High potential: {(predictions['potential_tier'] == 'High Potential').sum()}")
    print(f"   • Medium potential: {(predictions['potential_tier'] == 'Medium Potential').sum()}")
    print(f"   • Low potential: {(predictions['potential_tier'] == 'Low Potential').sum()}")
    
    print(f"\n🏆 Top 10 Predicted Breakouts for {predict_season}:")
    display_cols = ['player_name', 'position', 'recent_team', 'fantasy_points', 'breakout_probability', 'potential_tier']
    print(predictions[display_cols].head(10).to_string(index=False))

## Step 11: Prediction Analysis

Analyze the predictions in detail:

In [ ]:
if 'predictions' in locals() and not predictions.empty:
    print("📊 Analyzing predictions...")
    
    # Potential tier breakdown
    print("\n🎯 Potential Tier Distribution:")
    tier_counts = predictions['potential_tier'].value_counts()
    for tier, count in tier_counts.items():
        print(f"   • {tier}: {count} players ({count/len(predictions):.1%})")
    
    # Position breakdown
    print("\n🏈 Predictions by Position:")
    position_breakdown = predictions.groupby(['position', 'potential_tier']).size().unstack(fill_value=0)
    print(position_breakdown)
    
    # High potential players analysis
    high_potential = predictions[predictions['potential_tier'] == 'High Potential']
    if len(high_potential) > 0:
        print(f"\n⭐ High Potential Players Analysis:")
        print(f"   • Count: {len(high_potential)}")
        print(f"   • Avg 2024 fantasy points: {high_potential['fantasy_points'].mean():.1f}")
        print(f"   • Position breakdown: {dict(high_potential['position'].value_counts())}")
        
        print(f"\n📈 Fantasy Point Distribution for High-Potential Players:")
        bins = [0, 50, 100, 150, 200, 300, float('inf')]
        labels = ['0-50', '50-100', '100-150', '150-200', '200-300', '300+']
        high_potential_copy = high_potential.copy()
        high_potential_copy['fp_range'] = pd.cut(high_potential_copy['fantasy_points'], bins=bins, labels=labels)
        fp_dist = high_potential_copy['fp_range'].value_counts().sort_index()
        for range_val, count in fp_dist.items():
            print(f"   • {range_val}: {count} players")
    
    # Visualization
    plt.figure(figsize=(15, 10))
    
    # Subplot 1: Fantasy Points vs Breakout Probability
    plt.subplot(2, 2, 1)
    colors = {'Low Potential': 'red', 'Medium Potential': 'orange', 'High Potential': 'green'}
    for tier in predictions['potential_tier'].unique():
        tier_data = predictions[predictions['potential_tier'] == tier]
        plt.scatter(tier_data['fantasy_points'], tier_data['breakout_probability'], 
                   alpha=0.6, label=tier, color=colors.get(tier, 'blue'), s=20)
    plt.xlabel('2024 Fantasy Points')
    plt.ylabel('Breakout Probability')
    plt.title('Fantasy Points vs Breakout Probability')
    plt.legend()
    
    # Subplot 2: Probability distribution
    plt.subplot(2, 2, 2)
    plt.hist(predictions['breakout_probability'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    plt.xlabel('Breakout Probability')
    plt.ylabel('Count')
    plt.title('Distribution of Breakout Probabilities')
    
    # Subplot 3: Position breakdown
    plt.subplot(2, 2, 3)
    position_counts = predictions['position'].value_counts()
    plt.pie(position_counts.values, labels=position_counts.index, autopct='%1.1f%%')
    plt.title('Predictions by Position')
    
    # Subplot 4: Tier breakdown
    plt.subplot(2, 2, 4)
    tier_counts = predictions['potential_tier'].value_counts()
    colors_tier = ['red', 'orange', 'green']
    plt.pie(tier_counts.values, labels=tier_counts.index, autopct='%1.1f%%', colors=colors_tier)
    plt.title('Potential Tier Distribution')
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ No predictions available for analysis")

## Step 12: Position-Specific Analysis

Dive deeper into predictions by position:

In [ ]:
if 'predictions' in locals() and not predictions.empty:
    print("🏈 Position-Specific Analysis:")
    
    positions = ['QB', 'RB', 'WR', 'TE']
    
    for pos in positions:
        pos_data = predictions[predictions['position'] == pos]
        if len(pos_data) > 0:
            high_pot = pos_data[pos_data['potential_tier'] == 'High Potential']
            med_pot = pos_data[pos_data['potential_tier'] == 'Medium Potential']
            
            print(f"\n📊 {pos} Analysis:")
            print(f"   • Total players: {len(pos_data)}")
            print(f"   • High potential: {len(high_pot)} ({len(high_pot)/len(pos_data):.1%})")
            print(f"   • Medium potential: {len(med_pot)} ({len(med_pot)/len(pos_data):.1%})")
            print(f"   • Avg breakout probability: {pos_data['breakout_probability'].mean():.3f}")
            
            if len(high_pot) > 0:
                print(f"\n   🌟 Top 3 High-Potential {pos}s:")
                top_pos = high_pot.head(3)
                for _, player in top_pos.iterrows():
                    print(f"      • {player['player_name']} ({player['recent_team']}) - {player['breakout_probability']:.3f}")
            elif len(med_pot) > 0:
                print(f"\n   ⭐ Top 3 Medium-Potential {pos}s:")
                top_pos = med_pot.head(3)
                for _, player in top_pos.iterrows():
                    print(f"      • {player['player_name']} ({player['recent_team']}) - {player['breakout_probability']:.3f}")
else:
    print("❌ No predictions available for position analysis")

## Step 13: Save Results

Export predictions and analysis to CSV files:

In [ ]:
if 'predictions' in locals() and not predictions.empty:
    print("💾 Saving results...")
    
    # Save all predictions
    filename = f"player_breakout_predictions_{predict_season}_procedural.csv"
    predictions.to_csv(filename, index=False)
    print(f"   ✅ All predictions saved to '{filename}'")
    
    # Save high-potential players only
    high_potential_players = predictions[predictions['potential_tier'] == 'High Potential']
    if len(high_potential_players) > 0:
        high_filename = f"high_potential_players_{predict_season}_procedural.csv"
        high_potential_players.to_csv(high_filename, index=False)
        print(f"   ✅ High-potential players saved to '{high_filename}'")
    
    # Save model performance metrics
    import json
    metrics_filename = f"model_performance_{predict_season}_procedural.json"
    metrics_to_save = {
        'train_auc': float(model_results['train_auc']),
        'test_auc': float(model_results['test_auc']),
        'train_ap': float(model_results['train_ap']),
        'test_ap': float(model_results['test_ap']),
        'total_predictions': len(predictions),
        'high_potential_count': len(high_potential_players),
        'feature_count': len(selected_features),
        'top_features': dict(list(zip(sorted_features[:10], [float(x) for x in sorted_importances[:10]])))
    }
    
    with open(metrics_filename, 'w') as f:
        json.dump(metrics_to_save, f, indent=2)
    print(f"   ✅ Model metrics saved to '{metrics_filename}'")
    
    print(f"\n📈 Summary:")
    print(f"   • Total predictions: {len(predictions):,}")
    print(f"   • High potential players: {len(high_potential_players)}")
    print(f"   • Model AUC: {model_results['test_auc']:.3f}")
    print(f"   • Files saved: 3")
else:
    print("❌ No predictions available to save")

## Summary

This notebook successfully deconstructed the PlayerBreakoutPipeline abstraction into discrete procedural steps:

1. **Data Loading** - Retrieved NFL stats from nflverse
2. **Aggregation** - Converted weekly stats to season-level data
3. **Breakout Calculation** - Identified historical breakout patterns
4. **Feature Engineering** - Created predictive features
5. **Feature Selection** - Chose most relevant variables
6. **Data Preparation** - Prepared training/test sets with scaling
7. **Model Training** - Trained Random Forest classifier
8. **Model Evaluation** - Assessed performance metrics
9. **Feature Analysis** - Understood important predictors
10. **Predictions** - Generated breakout forecasts
11. **Analysis** - Analyzed predictions by tier and position
12. **Position Analysis** - Position-specific insights
13. **Export** - Saved results to CSV files

Each step is now transparent and can be customized independently, while maintaining the same functionality as the original abstracted pipeline.